In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset

In [7]:
train_df = pd.read_csv('../sign_lang_recognition/data/sign_mnist_train.csv')
test_df = pd.read_csv('../sign_lang_recognition/data/sign_mnist_test.csv')

train_labels = train_df['label'].values
train_images = train_df.drop('label', axis=1).values

test_labels = test_df['label'].values
test_images = test_df.drop('label', axis=1).values


In [8]:
class SignLanguageDataset(Dataset):

  def __init__(self, images, labels):
    self.images = images
    self.labels = labels

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx]
    image = image.reshape(28, 28)
    image = image / 255.0
    image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
    label = torch.tensor(self.labels[idx], dtype=torch.long)

    return image, label


In [9]:
train_dataset = SignLanguageDataset(train_images, train_labels)
test_dataset = SignLanguageDataset(test_images, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [10]:
def train_model(model, num_epochs = 20):

  optimizer = optim.Adam(model.parameters(), lr=0.001)
  loss_function = nn.CrossEntropyLoss()

  for epoch in range(num_epochs):
    model.train()

    correct = 0
    total = 0

    for imgs, labels in train_loader:
      preds = model(imgs)
      loss = loss_function(preds, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      predicted = preds.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

    accuracy = correct / total
    print("Epoch", epoch+1, "Train Accuracy:", accuracy)

    if accuracy == 1.0:
      print("Model has reached 100% accuracy!")
      break



In [11]:
def test_model(model, num_epochs = 20):

  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for imgs, labels in test_loader:
      outputs = model(imgs)
      predicted = outputs.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

  accuracy = correct / total
  print("Test Accuracy:", accuracy)
  return accuracy

In [12]:
def count_params(model):
  return sum(p.numel() for p in model.parameters())

In [13]:
class BaseCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size = 3, padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out


In [14]:
cnn = BaseCNN()
train_model(cnn)
cnn_test_accuracy = test_model(cnn)
print("CNN Parameters: ", count_params(cnn))

Epoch 1 Train Accuracy: 0.5725368785285012
Epoch 2 Train Accuracy: 0.8961937716262975
Epoch 3 Train Accuracy: 0.9734110362411218
Epoch 4 Train Accuracy: 0.9962484064833363
Epoch 5 Train Accuracy: 0.9992715352394828
Epoch 6 Train Accuracy: 0.9999635767619741
Epoch 7 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.8830172894590073
CNN Parameters:  27193


The test accuracy is not that good even though the training accuracy gets pretty high. I'm going to use Dropout to randomly deactivate some neurons to try to prevent overfitting in case there is some overfitting happening.

In [15]:
class DropoutCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)
    self.dropout1 = nn.Dropout2d(0.4)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)
    self.dropout2 = nn.Dropout2d(0.4)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.dropout1(out)
    out = self.pool2(self.act2(self.conv2(out)))
    out = self.dropout2(out)
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [16]:
cnn_dropout = DropoutCNN()
train_model(cnn_dropout)
cnn_dropout_test_accuracy = test_model(cnn_dropout)
print("CNN Parameters: ", count_params(cnn_dropout))

Epoch 1 Train Accuracy: 0.37450373338189763
Epoch 2 Train Accuracy: 0.767364778728829
Epoch 3 Train Accuracy: 0.8722272810052814
Epoch 4 Train Accuracy: 0.9235476233837188
Epoch 5 Train Accuracy: 0.9477326534328901
Epoch 6 Train Accuracy: 0.9636860316882171
Epoch 7 Train Accuracy: 0.9723547623383719
Epoch 8 Train Accuracy: 0.9773811691859406
Epoch 9 Train Accuracy: 0.981205609178656
Epoch 10 Train Accuracy: 0.9827353851757421
Epoch 11 Train Accuracy: 0.9841558914587507
Epoch 12 Train Accuracy: 0.9864505554543799
Epoch 13 Train Accuracy: 0.9873611364050264
Epoch 14 Train Accuracy: 0.9881988708796212
Epoch 15 Train Accuracy: 0.9897286468767074
Epoch 16 Train Accuracy: 0.9902021489710435
Epoch 17 Train Accuracy: 0.9912584228737935
Epoch 18 Train Accuracy: 0.9914405390639228
Epoch 19 Train Accuracy: 0.9906756510653797
Epoch 20 Train Accuracy: 0.9902021489710435
Test Accuracy: 0.8796709425543782
CNN Parameters:  27193


The test accuracy decreased, so there was probably no overfitting happening just underfitting. So I'm going to try to train with ReLU instead of Tanh to try to get a higer accuracy.

In [17]:
class ReLUCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 8, kernel_size = 3, padding=1)
    self.act1 = nn.ReLU()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8, 16, kernel_size = 3, padding=1)
    self.act2 = nn.ReLU()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7, 32)
    self.act3 = nn.ReLU()
    self.fc2 = nn.Linear(32, 25)

  def forward(self, x):
    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out


In [18]:
cnn_relu = ReLUCNN()
train_model(cnn_relu, 20)
cnn_test_accuracy = test_model(cnn_relu, 20)
print("CNN Parameters: ", count_params(cnn_relu))

Epoch 1 Train Accuracy: 0.41894008377344744
Epoch 2 Train Accuracy: 0.7751593516663632
Epoch 3 Train Accuracy: 0.8792569659442725
Epoch 4 Train Accuracy: 0.9342560553633218
Epoch 5 Train Accuracy: 0.9719541067200874
Epoch 6 Train Accuracy: 0.9867783645966126
Epoch 7 Train Accuracy: 0.9949007466763795
Epoch 8 Train Accuracy: 0.9978146057184484
Epoch 9 Train Accuracy: 0.9993443817155345
Epoch 10 Train Accuracy: 0.9942815516299399
Epoch 11 Train Accuracy: 0.9997814605718448
Epoch 12 Train Accuracy: 0.9997814605718448
Epoch 13 Train Accuracy: 0.9993443817155345
Epoch 14 Train Accuracy: 0.992678929156802
Epoch 15 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.88218070273285
CNN Parameters:  27193


Training with ReLU also decreased the test accuracy from the base model. I'm going to try to make the CNN model wider.

In [19]:
class WideCNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 32, kernel_size=3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(32,16, kernel_size=3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(16*7*7,64)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(64,25)

  def forward(self,x):

    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [20]:
cnn_wide = WideCNN()
train_model(cnn_wide)
cnn_test_accuracy = test_model(cnn_wide)
print("CNN Parameters: ", count_params(cnn_wide))

Epoch 1 Train Accuracy: 0.6745583682389364
Epoch 2 Train Accuracy: 0.980950646512475
Epoch 3 Train Accuracy: 0.9996721908577673
Epoch 4 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.9221974344673731
CNN Parameters:  56809


After trying to make the CNN wider the testing accuracy actually increased, but there were double the amount of parameters used. And by Epoch 4, the testing accuracy also reached 100%. Next, I'm going to try to make the CNN a deeper model to try to see what would happen.

In [21]:
class DeepCNN(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1,16,3,padding=1)
    self.act1 = nn.Tanh()

    self.conv2 = nn.Conv2d(16,16,3,padding=1)
    self.act2 = nn.Tanh()

    self.pool1 = nn.MaxPool2d(2)

    self.conv3 = nn.Conv2d(16,8,3,padding=1)
    self.act3 = nn.Tanh()

    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(8*7*7,32)
    self.fc2 = nn.Linear(32,25)

  def forward(self,x):

    out = x.view(x.size(0), 1, 28, 28)
    out = self.act1(self.conv1(out))
    out = self.act2(self.conv2(out))
    out = self.pool1(out)
    out = self.pool2(self.act3(self.conv3(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.fc1(out))

    return out

In [22]:
cnn_deep = DeepCNN()
train_model(cnn_deep)
cnn_test_accuracy = test_model(cnn_deep)
print("CNN Parameters: ", count_params(cnn_deep))

Epoch 1 Train Accuracy: 0.5725733017665271
Epoch 2 Train Accuracy: 0.9471863048625023
Epoch 3 Train Accuracy: 0.9946822072482243
Epoch 4 Train Accuracy: 0.9996357676197414
Epoch 5 Train Accuracy: 0.9998907302859225
Epoch 6 Train Accuracy: 0.9999635767619741
Epoch 7 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.9008644729503625
CNN Parameters:  17041


Making the CNN deeper did not really help the accuracy from the base CNN model, so maybe I'll focus on making the model wider instead of deeper.

In [23]:
class WideCNN2(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 64, kernel_size=3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(64,32, kernel_size=3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.fc1 = nn.Linear(32*7*7,128)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(128,25)

  def forward(self,x):

    x = x.view(x.size(0), 1, 28, 28)
    out = self.pool1(self.act1(self.conv1(x)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = out.view(out.size(0), -1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [24]:
cnn_wide2 = WideCNN2()
train_model(cnn_wide2)
cnn_test_accuracy = test_model(cnn_wide2)
print("CNN Parameters: ", count_params(cnn_wide2))

Epoch 1 Train Accuracy: 0.8233837188126024
Epoch 2 Train Accuracy: 0.9997814605718448
Epoch 3 Train Accuracy: 1.0
Model has reached 100% accuracy!
Test Accuracy: 0.931957612939208
CNN Parameters:  223161


The test accuracy is the highest so far, but it might be because this model is using the most number of parameters.

In [25]:
class GAPCNN(nn.Module):

  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1,8,3,padding=1)
    self.act1 = nn.Tanh()
    self.pool1 = nn.MaxPool2d(2)

    self.conv2 = nn.Conv2d(8,16,3,padding=1)
    self.act2 = nn.Tanh()
    self.pool2 = nn.MaxPool2d(2)

    self.gap = nn.AdaptiveAvgPool2d((3,3))

    self.fc1 = nn.Linear(16*3*3,32)
    self.act3 = nn.Tanh()
    self.fc2 = nn.Linear(32,25)

  def forward(self,x):

    out = x.view(x.size(0),1,28,28)
    out = self.pool1(self.act1(self.conv1(out)))
    out = self.pool2(self.act2(self.conv2(out)))
    out = self.gap(out)
    out = out.view(out.size(0),-1)
    out = self.fc2(self.act3(self.fc1(out)))

    return out

In [26]:
cnn_gap = GAPCNN()
train_model(cnn_gap, 20)
cnn_gap_test_accuracy = test_model(cnn_gap)
print("CNN Parameters:", count_params(cnn_gap))

Epoch 1 Train Accuracy: 0.1722454926242943
Epoch 2 Train Accuracy: 0.39624840648333637
Epoch 3 Train Accuracy: 0.6133673283554908
Epoch 4 Train Accuracy: 0.7479147696230195
Epoch 5 Train Accuracy: 0.8145328719723184
Epoch 6 Train Accuracy: 0.8544163176106356
Epoch 7 Train Accuracy: 0.8842105263157894
Epoch 8 Train Accuracy: 0.9122199963576761
Epoch 9 Train Accuracy: 0.9326898561282098
Epoch 10 Train Accuracy: 0.9494081223820797
Epoch 11 Train Accuracy: 0.9649608450191222
Epoch 12 Train Accuracy: 0.9755235840466218
Epoch 13 Train Accuracy: 0.9833181569841559
Epoch 14 Train Accuracy: 0.9880531779275178
Epoch 15 Train Accuracy: 0.9923511200145693
Epoch 16 Train Accuracy: 0.9941722819158623
Epoch 17 Train Accuracy: 0.9961027135312329
Epoch 18 Train Accuracy: 0.997122564195957
Epoch 19 Train Accuracy: 0.9978510289564743
Epoch 20 Train Accuracy: 0.9982516845747587
Test Accuracy: 0.9269380925822643
CNN Parameters: 6713


I was able to acheive a similar accuracy to the the wide cnn networks but using way less parameters with this Global Average Pooling Model, which also compressed the model from 16 feature maps of size 7x7 to 3x3 to have less parameters

- do augmentation
- train validation test split (k-fold)
- roc curve
- plots


In [27]:
from torchvision import transforms

class AugmentationSignLanguageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].reshape(28, 28, 1).astype(np.uint8)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.transform:
            image = self.transform(image)
        else:
            image = image / 255.0
            image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)

        return image, label

In [30]:
augmentation_strategy = transforms.Compose([
    transforms.ToTensor(),
    # Randomly rotate the image by up to 10 degrees in either direction
    transforms.RandomRotation(degrees=10), 
    # Randomly translate/shift the image vertically or horizontally by up to 10%
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)) 
])

# For testing/validation, we ONLY convert to tensor. We NEVER augment test data.
base_transform = transforms.Compose([
    transforms.ToTensor()
])



aug_train_dataset = AugmentationSignLanguageDataset(train_images, train_labels, transform=augmentation_strategy)
aug_train_loader = DataLoader(aug_train_dataset, batch_size=64, shuffle=True)

In [31]:
def augmented_train_model(model, num_epochs = 20):

  optimizer = optim.Adam(model.parameters(), lr=0.001)
  loss_function = nn.CrossEntropyLoss()

  for epoch in range(num_epochs):
    model.train()

    correct = 0
    total = 0

    for imgs, labels in aug_train_loader:
      preds = model(imgs)
      loss = loss_function(preds, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      predicted = preds.argmax(dim=1)
      correct += (predicted == labels).sum().item()
      total += labels.size(0)

    accuracy = correct / total
    print("Epoch", epoch+1, "Train Accuracy:", accuracy)

    if accuracy == 1.0:
      print("Model has reached 100% accuracy!")
      break



In [ ]:
cnn_wide2_aug = WideCNN2()
augmented_train_model(cnn_wide2_aug)
cnn_test_accuracy_aug = test_model(cnn_wide2_aug)

Epoch 1 Train Accuracy: 0.5558914587506829
Epoch 2 Train Accuracy: 0.8553997450373338
Epoch 3 Train Accuracy: 0.9315607357494081
Epoch 4 Train Accuracy: 0.9594245128391914
Epoch 5 Train Accuracy: 0.9710799490074667
Epoch 6 Train Accuracy: 0.9771990529958113
Epoch 7 Train Accuracy: 0.9804407211801129
Epoch 8 Train Accuracy: 0.9855399745037334
Epoch 9 Train Accuracy: 0.9874339828810781
Epoch 10 Train Accuracy: 0.986414132216354
Epoch 11 Train Accuracy: 0.9888544891640867
Epoch 12 Train Accuracy: 0.9869604807867419
Epoch 13 Train Accuracy: 0.9890730285922419
Epoch 14 Train Accuracy: 0.9904206883991987
Epoch 15 Train Accuracy: 0.989291568020397
Epoch 16 Train Accuracy: 0.9895829539246039
Epoch 17 Train Accuracy: 0.9920597341103624
Epoch 18 Train Accuracy: 0.9926425059187762
Epoch 19 Train Accuracy: 0.9929338918229831
Epoch 20 Train Accuracy: 0.9896558004006556
Test Accuracy: 0.9861963190184049


AttributeError: 'float' object has no attribute 'parameters'